# Chronoscope Raft chart demo

This notebook builds a temporary Chronoscope database from the repository's sample Raft trace and displays its timelines. Run the cells from top to bottom. The source trace is not modified.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY = "https://github.com/just-now/chronoscope.git"
BRANCH = "feat/chronoscope-event-relation"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_dir = Path("/content/chronoscope")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(repo_dir)],
            check=True,
        )
else:
    repo_dir = next(
        (path for path in (Path.cwd(), *Path.cwd().parents)
         if (path / "setup.py").exists() and (path / "chronoscope").is_dir()),
        None,
    )
    if repo_dir is None:
        raise RuntimeError("Start Jupyter from a Chronoscope checkout, or open this notebook in Colab.")

os.chdir(repo_dir)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--editable", str(repo_dir), "ipympl"],
    check=True,
)
print(f"Using Chronoscope from {repo_dir}")

In [ ]:
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

get_ipython().run_line_magic("matplotlib", "widget")

## Build the demo database

The database lives in a temporary directory and is removed when the notebook kernel stops.

In [ ]:
import tempfile

from chronoscope import db, db_options
from chronoscope.parser import parser

demo_directory = tempfile.TemporaryDirectory()
demo_db = Path(demo_directory.name) / "raft_demo.db"
config = repo_dir / "test" / "raft_chronoscope.yaml"
trace = repo_dir / "test" / "raft_trace.txt"

db.open(str(demo_db), db_options, create=True)
try:
    db.load(parser(str(config)), str(trace))
    db.mkidx()
    top_sm_id = (
        db.state_machine
        .select(db.state_machine.id)
        .where(db.state_machine.type == "top")
        .scalar()
    )
    raft_count = (
        db.state_machine
        .select()
        .where(db.state_machine.type == "raft")
        .count()
    )
finally:
    db.close()

if top_sm_id is None:
    raise RuntimeError("The sample trace contains no top state machine.")
print(f"Loaded {raft_count} Raft state machines; chart root: {top_sm_id:#x}")

## Display the Raft timelines

The toolbar supports pan, zoom, and saving the chart. To measure an interval, focus the chart and repeat `e` then click for each of its two endpoints. Press `d` to remove the latest marker.

In [ ]:
from chronoscope import chart

db.open(str(demo_db), db_options)
try:
    chart.plot(top_sm_id, figsize=(16, 7))
finally:
    db.close()

## Try another trace

Replace `config` and `trace` above with paths to files using the same Chronoscope formats, then rerun the last two code cells. In Colab, files can be uploaded through the Files panel.